# MinIo - create buckets

In [2]:
import os
import sys
import boto3
from dotenv import load_dotenv
from botocore.config import Config
from pathlib import Path

In [3]:
# working directory
os.getcwd()

'C:\\Projects\\fsb-projects\\mp7_pipeline_toko_bandung\\notebook'

In [3]:
# versi python
sys.version

'3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]'

In [4]:
# python execution
sys.executable

'C:\\Users\\Alvin Faisal\\AppData\\Local\\Programs\\Python\\Python313\\python.exe'

In [4]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\.venv\Scripts\python.exe
Python version: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]


In [5]:
# membaca konfigurasi minio
load_dotenv(dotenv_path="../.env")

True

In [6]:
# membuat koneksi ke minio
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9007",
    aws_access_key_id=os.getenv("MINIO_ROOT_USER"),
    aws_secret_access_key=os.getenv("MINIO_ROOT_PASSWORD"),
    region_name="us-east-1",
     config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"}
    )
)


print("Koneksi boto3 ke MinIO berhasil!")

Koneksi boto3 ke MinIO berhasil!


In [7]:
# memeriksa apakah ada buckets
response = s3.list_buckets()
response

{'ResponseMetadata': {'RequestId': '18E9BA0E5F6CFF61',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '558',
   'content-type': 'application/xml',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18E9BA0E5F6CFF61',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '1551',
   'x-ratelimit-remaining': '1551',
   'x-xss-protection': '1; mode=block',
   'date': 'Fri, 20 Nov 2026 10:28:17 GMT'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'toko-curated',
   'CreationDate': datetime.datetime(2026, 11, 19, 22, 34, 31, 160000, tzinfo=tzutc())},
  {'Name': 'toko-processed',
   'CreationDate': datetime.datetime(2026, 11, 19, 22, 34, 31, 143000, tzinfo=tzutc())},
  {'Name':

In [8]:
# menentukan nama buckets
buckets = [
    "toko-raw",
    "toko-processed",
    "toko-curated"
]

# memeriksa nama bucket yang sudah ada agar tidak terjadi duplikasi ketika menambahkan bucket dengan nama yang sama
# jika nama bucket sudah ada, masukkan ke existing_buckets
existing_buckets = [
    bucket["Name"] for bucket in s3.list_buckets()["Buckets"]
]

# periksa apakah nama bucket sudah tersedia, jika tidak buat bucket baru
# dilukan pemeriksaan agar tidak membuat bucket dengan nama yang sama, PRINSIP IDEMPOTENCY
for bucket_name in buckets:
    if bucket_name not in existing_buckets:
        s3.create_bucket(Bucket=bucket_name)
        print(f"Bucket berhasil dibuat: {bucket_name}")
    else:  
        print(f"Bucket sudah tersedia: {bucket_name}")

Bucket sudah tersedia: toko-raw
Bucket sudah tersedia: toko-processed
Bucket sudah tersedia: toko-curated


In [9]:
# periksa bucket
for bucket in response["Buckets"]:
    print(bucket["Name"])

toko-curated
toko-processed
toko-raw


## Bucket itu seperti wadah utama penyimpanan. Nantinya, di dalam bucket kita bisa membuat struktur folder menggunakan object prefix, misalnya:

In [23]:
toko-raw/
└── postgres/
    └── orders/
        └── ingest_date=2026-11-19/

SyntaxError: invalid character '└' (U+2514) (821725438.py, line 2)

# MinIo - ingest to toko-raw bucket

In [10]:
BASE_DIR = Path.cwd().parent
csv_path = BASE_DIR / "data" / "source" / "orders_seed.csv"

print(BASE_DIR)
print(csv_path)
print(csv_path.exists())

C:\Projects\fsb-projects\mp7_pipeline_toko_bandung
C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\data\source\orders_seed.csv
True


In [34]:
# Filename= Lokasi path file
# Bucket= Bucket tujuan MinIo
# Key= nama dan lokasi object didalam bucket

s3.upload_file(
    Filename=str(csv_path),
    Bucket="toko-raw",
    Key="orders/source/orders_seed.csv"
)

print("berhasil upload")

berhasil upload


In [11]:
# periksa apakah data benar benar masuk ke dalam bucket
response = s3.list_objects_v2(
    Bucket="toko-raw",
    Prefix="orders/"
)

# display(response["Contents"][0]["Key"])

for obj in response.get("Contents", []):
    print(obj["Key"])
    print(obj["Size"])

orders/source/orders_seed.csv
140902


## ingestion metadata

kenapa metadata penting?

Bayangkan nanti pipeline berjalan setiap hari. Kita perlu bisa menjawab:
- Data ini berasal dari mana?
- Kapan data diambil?
- Berapa banyak record yang masuk?
- File ini sudah pernah di-ingest sebelumnya?
- Apakah proses ingestion berhasil?


Kegunaan beberapa metadata
- ContentLength	Mengetahui ukuran file
- LastModified	Mengetahui kapan object terakhir dimodifikasi di storage
- ETag	Identifier untuk membantu memeriksa identitas/perubahan object
- ContentType	Informasi jenis konten object

In [11]:
metadata = s3.head_object(
    Bucket="toko-raw",
    Key="orders/source/orders_seed.csv"
)

display((metadata))
print(f"LastModified: {metadata["LastModified"]} ")
print(f"ContentLength: {metadata["ContentLength"]} ")
print(f"ETag: {metadata["ETag"]} ")
print(f"ContentType: {metadata["ContentType"]} ")

{'ResponseMetadata': {'RequestId': '18E9BA128186FDCA',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '140902',
   'content-type': 'binary/octet-stream',
   'etag': '"29c1eef2646dfcafa48d79332ed31c4f"',
   'last-modified': 'Thu, 19 Nov 2026 22:51:56 GMT',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18E9BA128186FDCA',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '1551',
   'x-ratelimit-remaining': '1551',
   'x-xss-protection': '1; mode=block',
   'date': 'Fri, 20 Nov 2026 10:28:35 GMT'},
  'RetryAttempts': 0},
 'AcceptRanges': 'bytes',
 'LastModified': datetime.datetime(2026, 11, 19, 22, 51, 56, tzinfo=tzutc()),
 'ContentLength': 140902,
 'ETag': '"29c1e

LastModified: 2026-11-19 22:51:56+00:00 
ContentLength: 140902 
ETag: "29c1eef2646dfcafa48d79332ed31c4f" 
ContentType: binary/octet-stream 


In [12]:
import hashlib
import json
import csv
from datetime import datetime, timezone

# Membaca jumlah baris CSV
with open(csv_path, "r", encoding="utf-8", newline="") as file:
    row_count = sum(1 for row in csv.DictReader(file))

# buat object md5 untuk hashing/checksum
sha256_hash = hashlib.sha256()

# melakukan checksum untuk file orders_seed.csv secara bertahap 
with open(csv_path, "rb") as file:
    # membaca file per chunk, baca sebanyak 8.192 bytes per iterasi, agar hemat memori karena tidak perlu memuat seluruh file ke ram
    for chunk in iter(lambda: file.read(8192), b""):
        sha256_hash.update(chunk)

ingestion_metadata = {
    "source": "local_csv",
    "source_file": csv_path.name,
    "target_bucket": "toko-raw",
    "target_key": "orders/source/orders_seed.csv",
    "ingestion_timestamp": datetime.now(timezone.utc).isoformat(),
    "row_count": row_count,
    "file_size_bytes": csv_path.stat().st_size,
    "sha256_checksum": sha256_hash.hexdigest()
}

print(csv_path.name)
# display(ingestion_metadata)

print(json.dumps(ingestion_metadata, indent=4))


orders_seed.csv
{
    "source": "local_csv",
    "source_file": "orders_seed.csv",
    "target_bucket": "toko-raw",
    "target_key": "orders/source/orders_seed.csv",
    "ingestion_timestamp": "2026-11-20T10:28:37.313215+00:00",
    "row_count": 2000,
    "file_size_bytes": 140902,
    "sha256_checksum": "109aec7e7e41499469fda400182b34c75ee8836cb5c5f96e5e2cedf42c4b6a25"
}


## upload metadata JSON ke MinIo

In [50]:

metadata_key = "orders/source/orders_seed.metadata.json"

s3.put_object(
    Bucket="toko-raw",
    Key=metadata_key,
    Body=json.dumps(ingestion_metadata, indent=4),
    ContentType="application/json"
)

print("Ingestion metadata berhasil disimpan!")

Ingestion metadata berhasil disimpan!


## periksa apakah sudah masuk ke MinIO

In [56]:

response = s3.get_object(
    Bucket="toko-raw",
    Key=metadata_key
)

stored_metadata = json.loads(
    response["Body"].read().decode("utf-8")
)

print(json.dumps(stored_metadata, indent=4))


{
    "source": "local_csv",
    "source_file": "orders_seed.csv",
    "target_bucket": "toko-raw",
    "target_key": "orders/source/orders_seed.csv",
    "ingestion_timestamp": "2026-11-20T08:55:46.717809+00:00",
    "row_count": 2000,
    "file_size_bytes": 140902,
    "md5_checksum": "109aec7e7e41499469fda400182b34c75ee8836cb5c5f96e5e2cedf42c4b6a25"
}


# Verifikasi hasil ingest toko-raw

bandingkan dengan cara menhitung hasil checksum nya
Source: orders_seed.csv di lokal
Target: orders/source/orders_seed.csv di MinIO.

In [62]:
# fungi checksum
def calculate_sha256_file(file_path):
    # bikin object sha256
    sha256_hash = hashlib.sha256()

    # baca sebagai bytes "rb"
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256_hash.update(chunk)
            
    return sha256_hash.hexdigest()


109aec7e7e41499469fda400182b34c75ee8836cb5c5f96e5e2cedf42c4b6a25


In [63]:
# hasil dari file lokal
source_checksum = calculate_sha256_file(csv_path)
print(source_checksum)

109aec7e7e41499469fda400182b34c75ee8836cb5c5f96e5e2cedf42c4b6a25


In [71]:
# hasil dari file di MinIO
# ambil dari minio
response = s3.get_object(
    Bucket="toko-raw",
    Key="orders/source/orders_seed.csv"
)

raw_checksum = hashlib.sha256()

body = response["Body"]
# print(body.read().decode("utf-8"))

for chunk in iter(lambda: body.read(8192), b""):
    raw_checksum.update(chunk)

minio_checksum = raw_checksum.hexdigest()

body.close()

print("MinIO SHA-256:", minio_checksum)


MinIO SHA-256: 109aec7e7e41499469fda400182b34c75ee8836cb5c5f96e5e2cedf42c4b6a25


In [72]:
# membandingkan hasilnya

if source_checksum == minio_checksum:
    print("PASS: File source dan MinIO identik!")
else:
    print("FAIL: File source dan MinIO berbeda!")

PASS: File source dan MinIO identik!
